# Dense MIMIC-III waveform SpO2 and lactate analysis

This notebook embeds the complete reproducible pipeline and runs it start-to-finish. It writes the standalone script, tables, figures, logs, and report to the output folder. The completed analysis used local Downloads. In Colab, put the required MIMIC-III v1.4 tables and supplied waveform artifacts under the configured Drive input folder. MIMIC data are not copied or uploaded by this notebook.

The upstream all-waveform candidate stay list and raw MIMIC-III waveform files were not present in the local inputs; the report preserves those limitations.

In [ ]:
from pathlib import Path
import base64, subprocess, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/physiograph_dense_waveform_analysis')
    INPUT_ROOT = Path('/content/drive/MyDrive/physiograph_dense_waveform_inputs')
else:
    INPUT_ROOT = Path.home() / 'Downloads'
    OUTPUT_ROOT = INPUT_ROOT / 'physiograph_dense_waveform_analysis'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SCRIPT = OUTPUT_ROOT / 'scripts' / 'run_dense_waveform_analysis.py'
REQUIREMENTS = OUTPUT_ROOT / 'requirements.txt'
SCRIPT.parent.mkdir(parents=True, exist_ok=True)
SCRIPT.write_bytes(base64.b64decode('...'))
REQUIREMENTS.write_text('numpy>=1.26\npandas>=2.1\nscipy>=1.11\nscikit-learn>=1.3\nmatplotlib>=3.7\n')
assert INPUT_ROOT.exists(), f'Missing input folder: {INPUT_ROOT}'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REQUIREMENTS)])
subprocess.check_call([
    sys.executable, str(SCRIPT),
    '--input-root', str(INPUT_ROOT),
    '--output-root', str(OUTPUT_ROOT),
])
print((OUTPUT_ROOT / '09_final_report' / 'FINAL_REPORT.md').read_text())


## Primary models with early SpO2 level and organ support
Reuse the fixed cohort and corrected endpoints. Add mean SpO2, vasopressor infusions, and documented mechanical ventilation during ICU hours 0–4. This step reconstructs only the added support covariates from the original validated clinical tables. Patient-level caches remain in your private output folder.


In [ ]:
import pandas as pd
SUPPORT_SCRIPT = OUTPUT_ROOT / "scripts" / "rerun_support_adjusted_models.py"
PRIVATE_CACHE = OUTPUT_ROOT / "10_support_adjustment_cache"
PRIVATE_CACHE.mkdir(parents=True, exist_ok=True)
SUPPORT_SCRIPT.write_bytes(base64.b64decode('...'))
REFERENCE_RESULTS = PRIVATE_CACHE / "reference_models.csv"
REFERENCE_RESULTS.write_bytes(base64.b64decode('YW5hbHlzaXMsc3BlY2lmaWNhdGlvbixlbmRwb2ludCxuLGV2ZW50cyxub25fZXZlbnRzLHBhdGllbnRzLGV4cG9zZWRfZXZlbnRzLGV4cG9zZWRfbix1bmV4cG9zZWRfZXZlbnRzLHVuZXhwb3NlZF9uLGV4cG9zZWRfcmlzayx1bmV4cG9zZWRfcmlzayxjcnVkZV9SUixjcnVkZV9SRCxja2RfcHJpb3JfY29kZV9wb3NpdGl2ZV9uLHBhcmFtZXRlcnNfaW5jbHVkaW5nX2ludGVyY2VwdCxpdGVyYXRpb25zLGNvbnZlcmdlZCxkZXNpZ25fcmFuayxmdWxsX3JhbmssZXZlbnRzX3Blcl9wYXJhbWV0ZXIsYWRqdXN0ZWRfUlIscm9idXN0X0NJX2xvdyxyb2J1c3RfQ0lfaGlnaCxyb2J1c3RfU0VfbG9nX1JSLHBfdmFsdWUsc3RhdHVzLHJlYXNvbixjb3ZhcmlhdGVzCnByaW1hcnlfbGFjdGF0ZV8xMmgsYmVmb3JlX0NLRF9leHBhbnNpb24sY29tcGxldGUtd2luZG93IGxhc3QtdmFsdWUgbGFjdGF0ZSByaXNlID49MC41IG1tb2wvTCwzODgsNTAsMzM4LDM3MywyNywxNTUsMjMsMjMzLDAuMTc0MTkzNTQ4Mzg3MDk2NzcsMC4wOTg3MTI0NDYzNTE5MzEzMywxLjc2NDY1NjM4MTQ4NjY3NiwwLjA3NTQ4MTEwMjAzNTE2NTQ1LDY5LDEwLDgsVHJ1ZSwxMCxUcnVlLDUuMCwxLjc4MTM5MzUzMTM0Nzc4MzgsMS4wNDM5OTk4NTA3NDkwMzY1LDMuMDM5NjIwMDgzNTE5MTEyMywwLjI3MjYyMDcxMTM3Mjk3NSwwLjAzNDE3OTY0MjU4Njg2ODMyLGF2YWlsYWJsZSwsYWdlX3llYXJzfHNleF9tYWxlfGJhc2VsaW5lX2xhY3RhdGVfbW1vbF9sfGNrZF9vbGRfNTg1fGVzcmRfbGFjdGF0ZXxsdW5nX2xhY3RhdGV8bGl2ZXJfcHJpb3J8ZGlhYmV0ZXNfcHJpb3IKcHJpbWFyeV9sYWN0YXRlXzEyaCxhZnRlcl9DS0RfZXhwYW5zaW9uLGNvbXBsZXRlLXdpbmRvdyBsYXN0LXZhbHVlIGxhY3RhdGUgcmlzZSA+PTAuNSBtbW9sL0wsMzg4LDUwLDMzOCwzNzMsMjcsMTU1LDIzLDIzMywwLjE3NDE5MzU0ODM4NzA5Njc3LDAuMDk4NzEyNDQ2MzUxOTMxMzMsMS43NjQ2NTYzODE0ODY2NzYsMC4wNzU0ODExMDIwMzUxNjU0NSw3NSwxMCw4LFRydWUsMTAsVHJ1ZSw1LjAsMS43NzIxNTIxNzczMzg1NDQyLDEuMDM5NTkyNzU1ODc0ODU0MywzLjAyMDkxNjkxMzc2MDk4NiwwLjI3MjEyNTM0Mjc0NDMwMjcsMC4wMzU0OTMwNDA1NzE2ODk1OSxhdmFpbGFibGUsLGFnZV95ZWFyc3xzZXhfbWFsZXxiYXNlbGluZV9sYWN0YXRlX21tb2xfbHxja2RfZXhwYW5kZWR8ZXNyZF9sYWN0YXRlfGx1bmdfbGFjdGF0ZXxsaXZlcl9wcmlvcnxkaWFiZXRlc19wcmlvcgpwcmltYXJ5X2xhY3RhdGVfMTJoLGFmdGVyX0NLRF9wbHVzX21lYW5fU3BPMixjb21wbGV0ZS13aW5kb3cgbGFzdC12YWx1ZSBsYWN0YXRlIHJpc2UgPj0wLjUgbW1vbC9MLDM4OCw1MCwzMzgsMzczLDI3LDE1NSwyMywyMzMsMC4xNzQxOTM1NDgzODcwOTY3NywwLjA5ODcxMjQ0NjM1MTkzMTMzLDEuNzY0NjU2MzgxNDg2Njc2LDAuMDc1NDgxMTAyMDM1MTY1NDUsNzUsMTEsOCxUcnVlLDExLFRydWUsNC41NDU0NTQ1NDU0NTQ1NDYsMS44MDA0NTkxODQ0MzM5OTgsMS4wMjU1MDkwOTIzMjM3MzMsMy4xNjEwMTg1NjA1MTU1MTM0LDAuMjg3MTY5Njc4NjI1ODIxNiwwLjA0MDU4NzkwNzEzMjI5MDM5LGF2YWlsYWJsZSwsYWdlX3llYXJzfHNleF9tYWxlfGJhc2VsaW5lX2xhY3RhdGVfbW1vbF9sfG1lYW5fYmlubmVkX3NwbzJ8Y2tkX2V4cGFuZGVkfGVzcmRfbGFjdGF0ZXxsdW5nX2xhY3RhdGV8bGl2ZXJfcHJpb3J8ZGlhYmV0ZXNfcHJpb3IKcHJpbWFyeV9sYWN0YXRlXzI0aCxiZWZvcmVfQ0tEX2V4cGFuc2lvbixjb21wbGV0ZS13aW5kb3cgbGFzdC12YWx1ZSBsYWN0YXRlIHJpc2UgPj0wLjUgbW1vbC9MLDQyMSw0OSwzNzIsNDA0LDI1LDE3NCwyNCwyNDcsMC4xNDM2NzgxNjA5MTk1NDAyMiwwLjA5NzE2NTk5MTkwMjgzNDAxLDEuNDc4Njg3NzM5NDYzNjAxMywwLjA0NjUxMjE2OTAxNjcwNjIxLDcyLDEwLDgsVHJ1ZSwxMCxUcnVlLDQuOSwxLjQ5NzQyMjUxOTE2NzI4OTksMC44NjY0Nzk1NjAwNDU1OTg2LDIuNTg3Nzk4MTQ4MTY1NTYzNiwwLjI3OTExMzMwMDM4MzAyNTgsMC4xNDgwMjkwNTg1NzcwMDE0NCxhdmFpbGFibGUsLGFnZV95ZWFyc3xzZXhfbWFsZXxiYXNlbGluZV9sYWN0YXRlX21tb2xfbHxja2Rfb2xkXzU4NXxlc3JkX2xhY3RhdGV8bHVuZ19sYWN0YXRlfGxpdmVyX3ByaW9yfGRpYWJldGVzX3ByaW9yCnByaW1hcnlfbGFjdGF0ZV8yNGgsYWZ0ZXJfQ0tEX2V4cGFuc2lvbixjb21wbGV0ZS13aW5kb3cgbGFzdC12YWx1ZSBsYWN0YXRlIHJpc2UgPj0wLjUgbW1vbC9MLDQyMSw0OSwzNzIsNDA0LDI1LDE3NCwyNCwyNDcsMC4xNDM2NzgxNjA5MTk1NDAyMiwwLjA5NzE2NTk5MTkwMjgzNDAxLDEuNDc4Njg3NzM5NDYzNjAxMywwLjA0NjUxMjE2OTAxNjcwNjIxLDc4LDEwLDgsVHJ1ZSwxMCxUcnVlLDQuOSwxLjQ3NzYxNDk1MDQ5Njk5ODIsMC44NTkzNDc1MTg4OTMwODI1LDIuNTQwNzAxOTc5MDM3MDY1LDAuMjc2NTM2Mjk4MjY1NTE5OCwwLjE1Nzk5MjU0MDE5MjQ5NTY1LGF2YWlsYWJsZSwsYWdlX3llYXJzfHNleF9tYWxlfGJhc2VsaW5lX2xhY3RhdGVfbW1vbF9sfGNrZF9leHBhbmRlZHxlc3JkX2xhY3RhdGV8bHVuZ19sYWN0YXRlfGxpdmVyX3ByaW9yfGRpYWJldGVzX3ByaW9yCnByaW1hcnlfbGFjdGF0ZV8yNGgsYWZ0ZXJfQ0tEX3BsdXNfbWVhbl9TcE8yLGNvbXBsZXRlLXdpbmRvdyBsYXN0LXZhbHVlIGxhY3RhdGUgcmlzZSA+PTAuNSBtbW9sL0wsNDIxLDQ5LDM3Miw0MDQsMjUsMTc0LDI0LDI0NywwLjE0MzY3ODE2MDkxOTU0MDIyLDAuMDk3MTY1OTkxOTAyODM0MDEsMS40Nzg2ODc3Mzk0NjM2MDEzLDAuMDQ2NTEyMTY5MDE2NzA2MjEsNzgsMTEsOCxUcnVlLDExLFRydWUsNC40NTQ1NDU0NTQ1NDU0NTQsMS40Mjc1NTEwNDQ4NzgwMTI5LDAuNzg0MDEwODA4OTEyODA5MywyLjU5OTMyODk0MTA4NzI2OTgsMC4zMDU3NjE2Nzk1ODY4Mjk2LDAuMjQ0MzUyNjc5NTk5Njg0MixhdmFpbGFibGUsLGFnZV95ZWFyc3xzZXhfbWFsZXxiYXNlbGluZV9sYWN0YXRlX21tb2xfbHxtZWFuX2Jpbm5lZF9zcG8yfGNrZF9leHBhbmRlZHxlc3JkX2xhY3RhdGV8bHVuZ19sYWN0YXRlfGxpdmVyX3ByaW9yfGRpYWJldGVzX3ByaW9yCmluX2hvc3BpdGFsX21vcnRhbGl0eSxiZWZvcmVfQ0tEX2V4cGFuc2lvbixpbmRleC1hZG1pc3Npb24gSE9TUElUQUxfRVhQSVJFX0ZMQUcsOTExLDE1OSw3NTIsODMxLDc5LDM0NSw4MCw1NjYsMC4yMjg5ODU1MDcyNDYzNzY4LDAuMTQxMzQyNzU2MTgzNzQ1NiwxLjYyMDA3MjQ2Mzc2ODExNTksMC4wODc2NDI3NTEwNjI2MzEyMiwxNzEsMTAsNyxUcnVlLDEwLFRydWUsMTUuOSwxLjUxMjY5MDUwNjIwMDMxMiwxLjE0OTczMTgxMzI4MjI5NDIsMS45OTAyMzE1ODM2NzM0NCwwLjEzOTk4MDE3NzYzNjE3MDQ2LDAuMDAzMTA4NzUxMDg5MDc4NDI0NixhdmFpbGFibGUsLGFnZV95ZWFyc3xzZXhfbWFsZXxiYXNlbGluZV9sYWN0YXRlX21tb2xfbHxja2Rfb2xkXzU4NXxlc3JkX21vcnRhbGl0eXxsdW5nX21vcnRhbGl0eXxsaXZlcl9wcmlvcnxkaWFiZXRlc19wcmlvcgppbl9ob3NwaXRhbF9tb3J0YWxpdHksYWZ0ZXJfQ0tEX2V4cGFuc2lvbixpbmRleC1hZG1pc3Npb24gSE9TUElUQUxfRVhQSVJFX0ZMQUcsOTExLDE1OSw3NTIsODMxLDc5LDM0NSw4MCw1NjYsMC4yMjg5ODU1MDcyNDYzNzY4LDAuMTQxMzQyNzU2MTgzNzQ1NiwxLjYyMDA3MjQ2Mzc2ODExNTksMC4wODc2NDI3NTEwNjI2MzEyMiwxODYsMTAsNyxUcnVlLDEwLFRydWUsMTUuOSwxLjUwODg1NDMwMzkzNjY0NSwxLjE0NzAzMzA1NzI0Mjg5MDYsMS45ODQ4MDg4MDQwMTE2OTMxLDAuMTM5ODgzNjU1ODA1NjU3ODcsMC4wMDMyNzUxMTE0MzY1MTkxNzM1LGF2YWlsYWJsZSwsYWdlX3llYXJzfHNleF9tYWxlfGJhc2VsaW5lX2xhY3RhdGVfbW1vbF9sfGNrZF9leHBhbmRlZHxlc3JkX21vcnRhbGl0eXxsdW5nX21vcnRhbGl0eXxsaXZlcl9wcmlvcnxkaWFiZXRlc19wcmlvcgppbl9ob3NwaXRhbF9tb3J0YWxpdHksYWZ0ZXJfQ0tEX3BsdXNfbWVhbl9TcE8yLGluZGV4LWFkbWlzc2lvbiBIT1NQSVRBTF9FWFBJUkVfRkxBRyw5MTEsMTU5LDc1Miw4MzEsNzksMzQ1LDgwLDU2NiwwLjIyODk4NTUwNzI0NjM3NjgsMC4xNDEzNDI3NTYxODM3NDU2LDEuNjIwMDcyNDYzNzY4MTE1OSwwLjA4NzY0Mjc1MTA2MjYzMTIyLDE4NiwxMSw3LFRydWUsMTEsVHJ1ZSwxNC40NTQ1NDU0NTQ1NDU0NTUsMS40MTI0NzkzNDgwMTY2MywxLjA0NzIxMjA1NDYxNzUzOSwxLjkwNTE1MTc3ODc0MDg2OSwwLjE1MjY2MDc3MzU4NDc5NTg1LDAuMDIzNjg2MTE2ODc2ODgyNjg4LGF2YWlsYWJsZSwsYWdlX3llYXJzfHNleF9tYWxlfGJhc2VsaW5lX2xhY3RhdGVfbW1vbF9sfG1lYW5fYmlubmVkX3NwbzJ8Y2tkX2V4cGFuZGVkfGVzcmRfbW9ydGFsaXR5fGx1bmdfbW9ydGFsaXR5fGxpdmVyX3ByaW9yfGRpYWJldGVzX3ByaW9yCg=='))
INPUT_MANIFEST = PRIVATE_CACHE / "original_input_manifest.csv"
INPUT_MANIFEST.write_bytes(base64.b64decode('ZmlsZW5hbWUsYnl0ZXMsc2hhMjU2LHJvd19jb3VudCxjb2x1bW5zLGluZmVycmVkX3RhYmxlLG5vdGVzDQpBRE1JU1NJT05TLmNzdi5neiwyNTI1MjU0LDY4ZjEyOGYxOWU5NTQzNDUwMDk4MmQ5NWU2MjM2MzhmMjJlYWRmMWM2YTA3ZjRiZmE2YzM5YzIxNzUxZDgwODUsNTg5NzYsIlJPV19JRCxTVUJKRUNUX0lELEhBRE1fSUQsQURNSVRUSU1FLERJU0NIVElNRSxERUFUSFRJTUUsQURNSVNTSU9OX1RZUEUsQURNSVNTSU9OX0xPQ0FUSU9OLERJU0NIQVJHRV9MT0NBVElPTixJTlNVUkFOQ0UsTEFOR1VBR0UsUkVMSUdJT04sTUFSSVRBTF9TVEFUVVMsRVRITklDSVRZLEVEUkVHVElNRSxFRE9VVFRJTUUsRElBR05PU0lTLEhPU1BJVEFMX0VYUElSRV9GTEFHLEhBU19DSEFSVEVWRU5UU19EQVRBIixBRE1JU1NJT05TLHNlbGVjdGVkIGZ1bGwgTUlNSUMtSUlJIHYxLjQgc291cmNlOyByb3cgY291bnQgcmVjb3JkZWQgd2hpbGUgYW5hbHlzaXMgc3RyZWFtZWQgdGhlIHRhYmxlDQpDSEFSVEVWRU5UUy5jc3YuZ3osNDI4NzAwNDIxMiw1OTYxYWMzNmQ4YmZiZDM5OTA4ODgxMWY5YTFkMzBiZDZmNDgyYjIzYTg0NjI4NTkzZDMxMjJiNWE2YmU4YmM0LDMzMDcxMjQ4MywiUk9XX0lELFNVQkpFQ1RfSUQsSEFETV9JRCxJQ1VTVEFZX0lELElURU1JRCxDSEFSVFRJTUUsU1RPUkVUSU1FLENHSUQsVkFMVUUsVkFMVUVOVU0sVkFMVUVVT00sV0FSTklORyxFUlJPUixSRVNVTFRTVEFUVVMsU1RPUFBFRCIsQ0hBUlRFVkVOVFMsc2VsZWN0ZWQgZnVsbCBNSU1JQy1JSUkgdjEuNCBzb3VyY2U7IHJvdyBjb3VudCByZWNvcmRlZCB3aGlsZSBhbmFseXNpcyBzdHJlYW1lZCB0aGUgdGFibGUNCkRfSUNEX0RJQUdOT1NFUy5jc3YuZ3osMjg0OTUwLDc4MjM2Y2U1MmIwNTBmYzU5YzBkOTFiNzc0OTcxYTRjM2ZhNDNhYjEzYjk5MjljMWNjNmI2N2ZkYzNlMjMwNGQsMTQ1NjcsIlJPV19JRCxJQ0Q5X0NPREUsU0hPUlRfVElUTEUsTE9OR19USVRMRSIsRF9JQ0RfRElBR05PU0VTLHNlbGVjdGVkIGZ1bGwgTUlNSUMtSUlJIHYxLjQgc291cmNlOyByb3cgY291bnQgcmVjb3JkZWQgd2hpbGUgYW5hbHlzaXMgc3RyZWFtZWQgdGhlIHRhYmxlDQpEX0lDRF9QUk9DRURVUkVTLmNzdi5neiw3NTg0OCwwZWVhMDBiNmFjZThjMjg1ZjBjZjAyZGYwMWVlMDQ0ZWFiYmI5OTllMmZkMzEzYzQ4Y2RhMDA0YTRmNjliMzM0LDM4ODIsIlJPV19JRCxJQ0Q5X0NPREUsU0hPUlRfVElUTEUsTE9OR19USVRMRSIsRF9JQ0RfUFJPQ0VEVVJFUyxzZWxlY3RlZCBmdWxsIE1JTUlDLUlJSSB2MS40IHNvdXJjZTsgcm93IGNvdW50IHJlY29yZGVkIHdoaWxlIGFuYWx5c2lzIHN0cmVhbWVkIHRoZSB0YWJsZQ0KRF9JVEVNUy5jc3YuZ3osMTg3OTIyLGViMThjNDA4NGY2YzhkYjQ5YWRkMjZjYmNmZWY0ZWM3NjI3YmJmNzJjMmI4ODdjMzZhYTBkNWYzZGNjYWY0YTAsMTI0ODcsIlJPV19JRCxJVEVNSUQsTEFCRUwsQUJCUkVWSUFUSU9OLERCU09VUkNFLExJTktTVE8sQ0FURUdPUlksVU5JVE5BTUUsUEFSQU1fVFlQRSxDT05DRVBUSUQiLERfSVRFTVMsc2VsZWN0ZWQgZnVsbCBNSU1JQy1JSUkgdjEuNCBzb3VyY2U7IHJvdyBjb3VudCByZWNvcmRlZCB3aGlsZSBhbmFseXNpcyBzdHJlYW1lZCB0aGUgdGFibGUNCkRfTEFCSVRFTVMuY3N2Lmd6LDExNDkyLGQwMzViY2E3MDA0MjNkYmE0ZWE3NGYxMzM4OTM2MjU5Njk0MWMwOTkxNDI1NTAwOGY4OWQxZjY1NmRiZWJiY2EsNzUzLCJST1dfSUQsSVRFTUlELExBQkVMLEZMVUlELENBVEVHT1JZLExPSU5DX0NPREUiLERfTEFCSVRFTVMsc2VsZWN0ZWQgZnVsbCBNSU1JQy1JSUkgdjEuNCBzb3VyY2U7IHJvdyBjb3VudCByZWNvcmRlZCB3aGlsZSBhbmFseXNpcyBzdHJlYW1lZCB0aGUgdGFibGUNCkRJQUdOT1NFU19JQ0QuY3N2Lmd6LDQ3MTc0NjMsMDJkMzk0MWFhM2M3YTA0M2FmNGYxMzhlNGZmYzlmMGE1MWNlY2EyMWM5MDVkZTFhYmQyYjk1MzIxMTkxMDQyMCw2NTEwNDcsIlJPV19JRCxTVUJKRUNUX0lELEhBRE1fSUQsU0VRX05VTSxJQ0Q5X0NPREUiLERJQUdOT1NFU19JQ0Qsc2VsZWN0ZWQgZnVsbCBNSU1JQy1JSUkgdjEuNCBzb3VyY2U7IHJvdyBjb3VudCByZWNvcmRlZCB3aGlsZSBhbmFseXNpcyBzdHJlYW1lZCB0aGUgdGFibGUNCklDVVNUQVlTLmNzdi5neiwxOTkwMTkzLDY1ZjhjODY4NmU3OTFiZWU4MGRmMDExNWZjMThmOWYzYzEwODdiODU4ODliYTljZTJkOWM0NDEyMDRlNDY3YWIsNjE1MzIsIlJPV19JRCxTVUJKRUNUX0lELEhBRE1fSUQsSUNVU1RBWV9JRCxEQlNPVVJDRSxGSVJTVF9DQVJFVU5JVCxMQVNUX0NBUkVVTklULEZJUlNUX1dBUkRJRCxMQVNUX1dBUkRJRCxJTlRJTUUsT1VUVElNRSxMT1MiLElDVVNUQVlTLHNlbGVjdGVkIGZ1bGwgTUlNSUMtSUlJIHYxLjQgc291cmNlOyByb3cgY291bnQgcmVjb3JkZWQgd2hpbGUgYW5hbHlzaXMgc3RyZWFtZWQgdGhlIHRhYmxlDQpJTlBVVEVWRU5UU19DVi5jc3YuZ3osNDIyMTA1Mzc2LDRmMTk1MDU2MzRiYTExYjM2OWU4ODU1ZGE1MTU1Njk2MjhkZGI2YjQxMDI1Yjc5NzU5NGY2OTdjMWM5MjY0YmQsMTc1Mjc5MzUsIlJPV19JRCxTVUJKRUNUX0lELEhBRE1fSUQsSUNVU1RBWV9JRCxDSEFSVFRJTUUsSVRFTUlELEFNT1VOVCxBTU9VTlRVT00sUkFURSxSQVRFVU9NLFNUT1JFVElNRSxDR0lELE9SREVSSUQsTElOS09SREVSSUQsU1RPUFBFRCxORVdCT1RUTEUsT1JJR0lOQUxBTU9VTlQsT1JJR0lOQUxBTU9VTlRVT00sT1JJR0lOQUxST1VURSxPUklHSU5BTFJBVEUsT1JJR0lOQUxSQVRFVU9NLE9SSUdJTkFMU0lURSIsSU5QVVRFVkVOVFNfQ1Ysc2VsZWN0ZWQgZnVsbCBNSU1JQy1JSUkgdjEuNCBzb3VyY2U7IHJvdyBjb3VudCByZWNvcmRlZCB3aGlsZSBhbmFseXNpcyBzdHJlYW1lZCB0aGUgdGFibGUNCklOUFVURVZFTlRTX01WLmNzdi5neiwxNTA5MDk3MzMsMWY1MjUzNDA3NGJmMDg0MGJiMDAyYWE4ZmExNmRhMTkzM2QwMWI4Y2YyY2M5MmYzZTMwZWM0MmUwNDY4NWI0MCwzNjE4OTkxLCJST1dfSUQsU1VCSkVDVF9JRCxIQURNX0lELElDVVNUQVlfSUQsU1RBUlRUSU1FLEVORFRJTUUsSVRFTUlELEFNT1VOVCxBTU9VTlRVT00sUkFURSxSQVRFVU9NLFNUT1JFVElNRSxDR0lELE9SREVSSUQsTElOS09SREVSSUQsT1JERVJDQVRFR09SWU5BTUUsU0VDT05EQVJZT1JERVJDQVRFR09SWU5BTUUsT1JERVJDT01QT05FTlRUWVBFREVTQ1JJUFRJT04sT1JERVJDQVRFR09SWURFU0NSSVBUSU9OLFBBVElFTlRXRUlHSFQsVE9UQUxBTU9VTlQsVE9UQUxBTU9VTlRVT00sSVNPUEVOQkFHLENPTlRJTlVFSU5ORVhUREVQVCxDQU5DRUxSRUFTT04sU1RBVFVTREVTQ1JJUFRJT04sQ09NTUVOVFNfRURJVEVEQlksQ09NTUVOVFNfQ0FOQ0VMRURCWSxDT01NRU5UU19EQVRFLE9SSUdJTkFMQU1PVU5ULE9SSUdJTkFMUkFURSIsSU5QVVRFVkVOVFNfTVYsc2VsZWN0ZWQgZnVsbCBNSU1JQy1JSUkgdjEuNCBzb3VyY2U7IHJvdyBjb3VudCByZWNvcmRlZCB3aGlsZSBhbmFseXNpcyBzdHJlYW1lZCB0aGUgdGFibGUNCkxBQkVWRU5UUy5jc3YuZ3osMzM1ODQzNzM1LDc5ZDg2MzE1ZGRlNmUxMTIxZjdhODhmNjJjMmI5MWI3YjgzNTM2NGFjMWU3NzcyNDhhNThlNGYxODAyMjRhZTYsMjc4NTQwNTUsIlJPV19JRCxTVUJKRUNUX0lELEhBRE1fSUQsSVRFTUlELENIQVJUVElNRSxWQUxVRSxWQUxVRU5VTSxWQUxVRVVPTSxGTEFHIixMQUJFVkVOVFMsc2VsZWN0ZWQgZnVsbCBNSU1JQy1JSUkgdjEuNCBzb3VyY2U7IHJvdyBjb3VudCByZWNvcmRlZCB3aGlsZSBhbmFseXNpcyBzdHJlYW1lZCB0aGUgdGFibGUNCk9VVFBVVEVWRU5UUy5jc3YuZ3osNTg0MzY1MjAsNzBmMTQzMjFkMjI4YjEyZDI0M2ExMDk3NGI3ZTQxZTQxNjkwMDMxZDZkYmMxNjM1NGVmNzhlM2QxZGZiNDQ0NSw0MzQ5MjE4LCJST1dfSUQsU1VCSkVDVF9JRCxIQURNX0lELElDVVNUQVlfSUQsQ0hBUlRUSU1FLElURU1JRCxWQUxVRSxWQUxVRVVPTSxTVE9SRVRJTUUsQ0dJRCxTVE9QUEVELE5FV0JPVFRMRSxJU0VSUk9SIixPVVRQVVRFVkVOVFMsc2VsZWN0ZWQgZnVsbCBNSU1JQy1JSUkgdjEuNCBzb3VyY2U7IHJvdyBjb3VudCByZWNvcmRlZCB3aGlsZSBhbmFseXNpcyBzdHJlYW1lZCB0aGUgdGFibGUNClBBVElFTlRTLmNzdi5neiw1NzE2MTUsMGExM2RiNjkwNDRkM2IwMWFiOWEzOTdmMzE5Mjc5MmMyYTFkMmU5OGM5Y2I0MWM0Zjk0ZTA0MDM5OTZkYzdjNSw0NjUyMCwiUk9XX0lELFNVQkpFQ1RfSUQsR0VOREVSLERPQixET0QsRE9EX0hPU1AsRE9EX1NTTixFWFBJUkVfRkxBRyIsUEFUSUVOVFMsc2VsZWN0ZWQgZnVsbCBNSU1JQy1JSUkgdjEuNCBzb3VyY2U7IHJvdyBjb3VudCByZWNvcmRlZCB3aGlsZSBhbmFseXNpcyBzdHJlYW1lZCB0aGUgdGFibGUNClBST0NFRFVSRUVWRU5UU19NVi5jc3YuZ3osNzgxNDMyMSw2YTExMDdiNmE2ZjExYWIzN2FjNjJiM2VmMzkwMWRiMmM1Zjk3M2M1Nzk0OTUxMTQxZWZkMGUyNjhlNGE5YzllLDI1ODA2NiwiUk9XX0lELFNVQkpFQ1RfSUQsSEFETV9JRCxJQ1VTVEFZX0lELFNUQVJUVElNRSxFTkRUSU1FLElURU1JRCxWQUxVRSxWQUxVRVVPTSxMT0NBVElPTixMT0NBVElPTkNBVEVHT1JZLFNUT1JFVElNRSxDR0lELE9SREVSSUQsTElOS09SREVSSUQsT1JERVJDQVRFR09SWU5BTUUsU0VDT05EQVJZT1JERVJDQVRFR09SWU5BTUUsT1JERVJDQVRFR09SWURFU0NSSVBUSU9OLElTT1BFTkJBRyxDT05USU5VRUlOTkVYVERFUFQsQ0FOQ0VMUkVBU09OLFNUQVRVU0RFU0NSSVBUSU9OLENPTU1FTlRTX0VESVRFREJZLENPTU1FTlRTX0NBTkNFTEVEQlksQ09NTUVOVFNfREFURSIsUFJPQ0VEVVJFRVZFTlRTX01WLHNlbGVjdGVkIGZ1bGwgTUlNSUMtSUlJIHYxLjQgc291cmNlOyByb3cgY291bnQgcmVjb3JkZWQgd2hpbGUgYW5hbHlzaXMgc3RyZWFtZWQgdGhlIHRhYmxlDQpQUk9DRURVUkVTX0lDRC5jc3YuZ3osMTc5NTAwNCw0ODQ2NGJmMWQwNTYyOWU2MDU5ZjNmMmZjNzVhMzgxNjU2MDE3NTdhYmNkYmU4NzEyNzgxNTdkYjExZDhkYjYwLDI0MDA5NSwiUk9XX0lELFNVQkpFQ1RfSUQsSEFETV9JRCxTRVFfTlVNLElDRDlfQ09ERSIsUFJPQ0VEVVJFU19JQ0Qsc2VsZWN0ZWQgZnVsbCBNSU1JQy1JSUkgdjEuNCBzb3VyY2U7IHJvdyBjb3VudCByZWNvcmRlZCB3aGlsZSBhbmFseXNpcyBzdHJlYW1lZCB0aGUgdGFibGUNCm1pbWljM193YXZlZm9ybV9oZl9yZWNvcmRfbWFuaWZlc3QuY3N2LDU1ODY5Nyw0MDM3YWFmMzVhNzRhZmEyMzFmMzc4M2RkMmRjNmU4NDllODU1ZGJlNjA2MTUyM2IxZDU4MTYwZDE4M2Y4OTExLDE3MjgsIlNVQkpFQ1RfSUQsSEFETV9JRCxJQ1VTVEFZX0lELElOVElNRSxyZWNvcmQscmVjb3JkX3BhdGgscmVjb3JkX3N0YXJ0LHJlY29yZF9lbmQsZnMsbnNhbXAsb3ZlcmxhcF9zdGFydCxvdmVybGFwX2VuZCxkYXRhZmlsZSxkYXRhX3BhdGgsaGVhZGVyX3BhdGgsb3ZlcmxhcF9ob3VycyIsd2F2ZWZvcm1fbWFuaWZlc3Qsc2VsZWN0ZWQgd2F2ZWZvcm0gc291cmNlOyByb3cgY291bnQgcmVjb3JkZWQgd2hpbGUgYW5hbHlzaXMgc3RyZWFtZWQgdGhlIHRhYmxlDQptaW1pYzNfd2F2ZWZvcm1fc3BvMl8xNW1pbi5jc3YuZ3osNDMwNDE1LGZhMTY5ZDZmYzhkYjJiMzAwNGY4YjVhMjdmOTBjZDJiMWI0NWU4MzE3MWJhMjQyNzk0ZDM1MzUyN2FhNjg4MjMsMTg1MDksImJpbl9pbmRleCxzcG8yX21lZGlhbixuX3Jhd19pbl9iaW4sZmlyc3RfdGltZSxsYXN0X3RpbWUscmVwcmVzZW50YXRpdmVfdGltZSxTVUJKRUNUX0lELEhBRE1fSUQsSUNVU1RBWV9JRCxJTlRJTUUscmF3X3NwbzJfY291bnQscmF3X3NwbzJfbWVhbixyYXdfc3BvMl9taW4scmF3X2JlbG93OTBfZnJhY3Rpb24iLHdhdmVmb3JtXzE1bWluLHNlbGVjdGVkIHdhdmVmb3JtIHNvdXJjZTsgcm93IGNvdW50IHJlY29yZGVkIHdoaWxlIGFuYWx5c2lzIHN0cmVhbWVkIHRoZSB0YWJsZQ0KbWltaWMzX3dhdmVmb3JtX3NwbzJfZmVhdHVyZXMuY3N2LDE4NDMyOSxhODUwNzE4YzE5ZWIxMjViOTE4NDZmNTI4ZGIyNzVkZDM0MDZlNzdkZjZiYzI1NGU5MjdlODc4MjdmYzcyNDZjLDE2NTQsIlNVQkpFQ1RfSUQsSEFETV9JRCxJQ1VTVEFZX0lELG9ic2VydmVkXzE1bWluX2JpbnMsbWlzc2luZ18xNW1pbl9iaW5zLHF1YWxpZnlpbmdfdHJhbnNpdGlvbnNfbGUzMG1pbixkeW5hbWljc19lbGlnaWJsZSxleHBvc3VyZV9hYnNfanVtcF9nZTQscm1zc2RfYmlubmVkLGp1bXBfZnJhY3Rpb25fZ2UzLGRyb3BfY291bnRfZ2UzLHJhd19zcG8yX2NvdW50LHJhd19zcG8yX21lYW4scmF3X3NwbzJfbWluLHJhd19iZWxvdzkwX2ZyYWN0aW9uIix3YXZlZm9ybV9mZWF0dXJlcyxzZWxlY3RlZCB3YXZlZm9ybSBzb3VyY2U7IHJvdyBjb3VudCByZWNvcmRlZCB3aGlsZSBhbmFseXNpcyBzdHJlYW1lZCB0aGUgdGFibGUNCg=='))
inventory = pd.read_csv(OUTPUT_ROOT / "data_inventory.csv")
selected = inventory.loc[inventory.inferred_table.eq("ADMISSIONS") & inventory.notes.fillna("").str.contains("selected full MIMIC-III v1.4", regex=False)]
assert len(selected) == 1, "Set CLINICAL_ROOT to the validated full MIMIC-III clinical-table directory."
CLINICAL_ROOT = Path(selected.iloc[0].full_path).parent
SUPPORT_RESULTS = OUTPUT_ROOT / "support_adjusted_models"
subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb==1.5.6"])
subprocess.check_call([sys.executable, str(SUPPORT_SCRIPT), "--source-output-root", str(OUTPUT_ROOT), "--clinical-root", str(CLINICAL_ROOT), "--private-cache-root", str(PRIVATE_CACHE), "--output-root", str(SUPPORT_RESULTS), "--reference-results", str(REFERENCE_RESULTS), "--input-manifest", str(INPUT_MANIFEST)])
print((SUPPORT_RESULTS / "REPORT.md").read_text())


## Hour-16 Kapur/CSWG minimum evidenced SCAI distribution
Uses latest valid physiology during ICU hours 12–16 and time-specific treatment documentation. The fixed SpO2 cohort and original outcomes are preserved. Stage A cannot be confirmed because OHCA and full examination information are unavailable; all assigned B–E stages are lower bounds. Results include the full cohort and the two primary-model samples, missingness, early departures/deaths, exposure strata, and physiology freshness sensitivities. Patient-linked caches remain in the supplied private output directory.


In [ ]:
SCAI_SCRIPT = OUTPUT_ROOT / "scripts" / "calculate_scai_hour16.py"
SCAI_SCRIPT.write_bytes(base64.b64decode('...'))
SCAI_RESULTS = OUTPUT_ROOT / "scai_hour16"
subprocess.check_call([sys.executable, str(SCAI_SCRIPT), "--source-output-root", str(OUTPUT_ROOT), "--clinical-root", str(CLINICAL_ROOT), "--private-cache-root", str(OUTPUT_ROOT / "11_scai_hour16_cache"), "--output-root", str(SCAI_RESULTS), "--input-manifest", str(INPUT_MANIFEST)])
print((SCAI_RESULTS / "REPORT.md").read_text())


## Hour-16 SCAI proxy graphs

Generate overall and early-instability-stratified graphs from the aggregate SCAI tables above. All sample stays remain in percentage denominators. Stage A is unavailable, and B–E are minimum evidenced proxy stages. Figures are saved as PNG, PDF, and SVG; no clinical processing or model estimation is repeated.


In [ ]:
from IPython.display import display, Image

SCAI_PLOT_SCRIPT = OUTPUT_ROOT / "scripts" / "plot_scai_hour16.py"
SCAI_PLOT_SCRIPT.parent.mkdir(parents=True, exist_ok=True)
SCAI_PLOT_SCRIPT.write_bytes(base64.b64decode("IiIiRXhwb3J0IGhvdXItMTYgU0NBSSBwcm94eSBmaWd1cmVzIGZyb20gYWdncmVnYXRlIHJlc3VsdHMgb25seS4iIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG1hdHBsb3RsaWIKbWF0cGxvdGxpYi51c2UoIkFnZyIpCmltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKZnJvbSBtYXRwbG90bGliLnRpY2tlciBpbXBvcnQgUGVyY2VudEZvcm1hdHRlcgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKClBPUFVMQVRJT05TID0gewogICAgImFsbF9zcG8yX2VsaWdpYmxlIjogIkZ1bGwgU3BP4oKCLWVsaWdpYmxlIGNvaG9ydCIsCiAgICAicHJpbWFyeV9sYWN0YXRlX21vZGVsIjogIlByaW1hcnkgbGFjdGF0ZSBtb2RlbCBzYW1wbGUiLAogICAgIm1vcnRhbGl0eV9tb2RlbCI6ICJNb3J0YWxpdHkgbW9kZWwgc2FtcGxlIiwKfQpDQVRFR09SSUVTID0gWyJCIiwgIkMiLCAiRCIsICJFIiwgInVuY2xhc3NpZmllZCIsICJkaWVkX2J5X2hvdXIxNiIsICJkZXBhcnR1cmUiXQpMQUJFTFMgPSBbIlN0YWdlIEIiLCAiU3RhZ2UgQyIsICJTdGFnZSBEIiwgIlN0YWdlIEUiLCAiVW5jbGFzc2lmaWVkIiwgIkRpZWQgYnkgaG91ciAxNiIsICJMZWZ0IElDVSAvIGRpc2NoYXJnZWQiXQpDT0xPUlMgPSBbIiMyNjc2QUQiLCAiI0NGOTQyQiIsICIjQzY1NDQ4IiwgIiM4MDY0QTIiLCAiIzlBQTVBRiIsICIjMzc0NDUxIiwgIiNDQUQyRDkiXQoKCmRlZiBjb3VudHNfZm9yKGZyYW1lLCBwb3B1bGF0aW9uLCBncm91cCk6CiAgICByb3dzID0gZnJhbWVbKGZyYW1lLnBvcHVsYXRpb24gPT0gcG9wdWxhdGlvbikgJiAoZnJhbWUuZXhwb3N1cmVfZ3JvdXAgPT0gZ3JvdXApXQogICAgZXhwZWN0ZWQgPSB7IkEiLCAiQiIsICJDIiwgIkQiLCAiRSIsICJ1bmNsYXNzaWZpZWQiLCAiZGllZF9ieV9ob3VyMTYiLCAiZGlzY2hhcmdlZF9ieV9ob3VyMTYiLCAibGVmdF9pY3VfYnlfaG91cjE2In0KICAgIGlmIGxlbihyb3dzKSAhPSBsZW4oZXhwZWN0ZWQpIG9yIHNldChyb3dzLmNhdGVnb3J5KSAhPSBleHBlY3RlZCBvciByb3dzLmNvaG9ydF9uLm51bmlxdWUoKSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJJbmNvbXBsZXRlIG9yIGR1cGxpY2F0ZWQgY2F0ZWdvcmllczoge3BvcHVsYXRpb259L3tncm91cH0iKQogICAgbiA9IGludChyb3dzLmNvaG9ydF9uLmlsb2NbMF0pCiAgICBjb3VudHMgPSByb3dzLnNldF9pbmRleCgiY2F0ZWdvcnkiKVsibiJdLnRvX2RpY3QoKQogICAgaWYgYW55KHYgPCAwIG9yIGludCh2KSAhPSB2IGZvciB2IGluIGNvdW50cy52YWx1ZXMoKSkgb3Igc3VtKGNvdW50cy52YWx1ZXMoKSkgIT0gbjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQ2F0ZWdvcnkgY291bnRzIGRvIG5vdCByZWNvbmNpbGU6IHtwb3B1bGF0aW9ufS97Z3JvdXB9IikKICAgIGlmIGNvdW50c1siQSJdICE9IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU3RhZ2UgQSBhdmFpbGFiaWxpdHkgaGFzIGNoYW5nZWQ7IHJldmlzZSB0aGUgZmlndXJlIGxhYmVscy4iKQogICAgaWYgbm90IG5wLmFsbGNsb3NlKHJvd3MucGVyY2VudF9vZl9jb2hvcnQsIHJvd3MubiAvIG4gKiAxMDAsIHJ0b2w9MCwgYXRvbD0xZS04KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTdG9yZWQgcGVyY2VudGFnZXMgZGlzYWdyZWUgd2l0aCBjYXRlZ29yeSBjb3VudHMuIikKICAgIGNvdW50c1siZGVwYXJ0dXJlIl0gPSBjb3VudHNbImRpc2NoYXJnZWRfYnlfaG91cjE2Il0gKyBjb3VudHNbImxlZnRfaWN1X2J5X2hvdXIxNiJdCiAgICByZXR1cm4gbiwgbnAuYXJyYXkoW2NvdW50c1tjXSBmb3IgYyBpbiBDQVRFR09SSUVTXSwgZHR5cGU9aW50KQoKCmRlZiBzdHlsZV9heGlzKGF4LCBpbmRleCk6CiAgICBheC5zZXRfeGxpbSgwLCA3NCkKICAgIGF4LnNldF94dGlja3MoWzAsIDIwLCA0MCwgNjBdKQogICAgYXgueGF4aXMuc2V0X21ham9yX2Zvcm1hdHRlcihQZXJjZW50Rm9ybWF0dGVyKDEwMCwgZGVjaW1hbHM9MCkpCiAgICBheC5ncmlkKGF4aXM9IngiLCBjb2xvcj0iI0U1RTlFRCIsIGxpbmV3aWR0aD0wLjcpCiAgICBheC5zZXRfYXhpc2JlbG93KFRydWUpCiAgICBheC5zZXRfeXRpY2tzKG5wLmFyYW5nZShsZW4oQ0FURUdPUklFUykpLCBMQUJFTFMgaWYgaW5kZXggPT0gMCBlbHNlIFsiIl0gKiBsZW4oQ0FURUdPUklFUykpCiAgICBheC5zZXRfeWxpbShsZW4oQ0FURUdPUklFUykgLSAwLjUsIC0wLjcpCiAgICBheC50aWNrX3BhcmFtcyhheGlzPSJib3RoIiwgbGVuZ3RoPTAsIHBhZD04KQogICAgZm9yIHNwaW5lIGluIGF4LnNwaW5lcy52YWx1ZXMoKToKICAgICAgICBzcGluZS5zZXRfdmlzaWJsZShGYWxzZSkKICAgIGF4LnNldF94bGFiZWwoIlBlcmNlbnRhZ2Ugb2Ygc2FtcGxlIiwgbGFiZWxwYWQ9MTIpCgoKZGVmIGZvb3RlcihmaWcsIHN0cmF0aWZpZWQ9RmFsc2UpOgogICAgZGVub21pbmF0b3IgPSAiUGVyY2VudGFnZXMgdXNlIHRoZSBmdWxsIHNhbXBsZSBpbiBlYWNoIHBhbmVsLiIgaWYgbm90IHN0cmF0aWZpZWQgZWxzZSAiUGVyY2VudGFnZXMgdXNlIGVhY2ggZXhwb3N1cmUgZ3JvdXAncyBmdWxsIHNhbXBsZTsgbGFiZWxzIHNob3cgY291bnQgKHBlcmNlbnRhZ2UpLiIKICAgIGZpZy50ZXh0KDAuMDM1LCAwLjA5MiwgIlN0YWdlIEEgY2Fubm90IGJlIGRldGVybWluZWQgZnJvbSB0aGUgYXZhaWxhYmxlIGRhdGE7IGl0IGlzIG5vdCBhIHplcm8tcHJldmFsZW5jZSBjYXRlZ29yeS4iLCBmb250c2l6ZT0xMSwgd2VpZ2h0PSJib2xkIiwgY29sb3I9IiMyNjM1NDIiKQogICAgZmlnLnRleHQoMC4wMzUsIDAuMDYxLCAiQuKAk0UgYXJlIG1pbmltdW0gZXZpZGVuY2VkIFNDQUkgcHJveHkgc3RhZ2VzLiBVbmNsYXNzaWZpZWQgc3RheXMgbWF5IGluY2x1ZGUgc3RhZ2UgQSBvciBoaWdoZXIgc3RhZ2VzLiIsIGZvbnRzaXplPTEwLCBjb2xvcj0iIzUyNjA2RCIpCiAgICBmaWcudGV4dCgwLjAzNSwgMC4wMzQsIGRlbm9taW5hdG9yICsgIiBDb3VudHMgYXJlIElDVSBzdGF5czsgdGhlIG1vZGVsIHNhbXBsZXMgb3ZlcmxhcCBhbmQgYXJlIHN1YnNldHMgb2YgdGhlIGZ1bGwgY29ob3J0LiIsIGZvbnRzaXplPTksIGNvbG9yPSIjNTI2MDZEIikKICAgIGZpZy50ZXh0KDAuMDM1LCAwLjAxMSwgIlBoeXNpb2xvZ3k6IGxhdGVzdCB2YWxpZCB2YWx1ZSBpbiBob3VycyAxMuKAkzE2OyB0cmVhdG1lbnQgZXZpZGVuY2UgYXQgaG91ciAxNi4gSUNVIGRlcGFydHVyZS9kaXNjaGFyZ2UgY2F0ZWdvcmllcyBjb21iaW5lZCBmb3IgZGlzcGxheS4iLCBmb250c2l6ZT05LCBjb2xvcj0iIzUyNjA2RCIpCgoKZGVmIGV4cG9ydChmaWcsIHJvb3QsIHN0ZW0pOgogICAgZm9yIGV4dGVuc2lvbiBpbiAoInBuZyIsICJwZGYiLCAic3ZnIik6CiAgICAgICAgZmlnLnNhdmVmaWcocm9vdCAvIGYie3N0ZW19LntleHRlbnNpb259IiwgZHBpPTMwMCwgZmFjZWNvbG9yPSJ3aGl0ZSIpCiAgICBwbHQuY2xvc2UoZmlnKQoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yZXN1bHRzLWRpciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBzb3VyY2UgPSBhcmdzLnJlc3VsdHNfZGlyIC8gInN0YWdlX2Rpc3RyaWJ1dGlvbi5jc3YiCiAgICBmcmFtZSA9IHBkLnJlYWRfY3N2KHNvdXJjZSkKICAgIGZyYW1lID0gZnJhbWVbZnJhbWUubWVhc3VyZW1lbnRfbG9va2JhY2tfbWludXRlcyA9PSAyNDBdLmNvcHkoKQogICAgb3V0cHV0ID0gYXJncy5yZXN1bHRzX2RpciAvICJmaWd1cmVzIgogICAgb3V0cHV0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBsdC5yY1BhcmFtcy51cGRhdGUoeyJmb250LmZhbWlseSI6ICJEZWphVnUgU2FucyIsICJmb250LnNpemUiOiAxMSwgInN2Zy5mb250dHlwZSI6ICJub25lIiwgInBkZi5mb250dHlwZSI6IDQyfSkKCiAgICBmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMoMSwgMywgZmlnc2l6ZT0oMTcsIDguNSkpCiAgICBmaWcuc3VicGxvdHNfYWRqdXN0KGxlZnQ9MC4xNDUsIHJpZ2h0PTAuOTksIHRvcD0wLjc5LCBib3R0b209MC4yMSwgd3NwYWNlPTAuMTEpCiAgICBmaWcudGV4dCgwLjAzNSwgMC45NCwgIkhvdXItMTYgU0NBSSBwcm94eSBkaXN0cmlidXRpb24iLCBmb250c2l6ZT0yMywgd2VpZ2h0PSJib2xkIiwgY29sb3I9IiMyNjM1NDIiKQogICAgZmlnLnRleHQoMC4wMzUsIDAuODkxLCAiQWxsIHN0YXlzIHJlbWFpbiBpbiB0aGUgZGVub21pbmF0b3IsIGluY2x1ZGluZyB1bmNsYXNzaWZpZWQgc3RheXMgYW5kIHRob3NlIG5vIGxvbmdlciBpbiB0aGUgSUNVLiIsIGZvbnRzaXplPTEyLCBjb2xvcj0iIzUyNjA2RCIpCiAgICBmb3IgaSwgKHBvcHVsYXRpb24sIHRpdGxlKSBpbiBlbnVtZXJhdGUoUE9QVUxBVElPTlMuaXRlbXMoKSk6CiAgICAgICAgbiwgY291bnRzID0gY291bnRzX2ZvcihmcmFtZSwgcG9wdWxhdGlvbiwgImFsbCIpCiAgICAgICAgcmF0ZXMgPSBjb3VudHMgLyBuICogMTAwCiAgICAgICAgYXggPSBheGVzW2ldCiAgICAgICAgc3R5bGVfYXhpcyhheCwgaSkKICAgICAgICBheC5zZXRfdGl0bGUoZiJ7dGl0bGV9XG5uID0ge246LH0gc3RheXMiLCBsb2M9ImxlZnQiLCBmb250c2l6ZT0xMiwgd2VpZ2h0PSJib2xkIiwgcGFkPTIxKQogICAgICAgIGF4LmJhcmgobnAuYXJhbmdlKGxlbihjb3VudHMpKSwgcmF0ZXMsIGhlaWdodD0wLjYsIGNvbG9yPUNPTE9SUykKICAgICAgICBmb3IgeSwgY291bnQsIHJhdGUgaW4gemlwKHJhbmdlKGxlbihjb3VudHMpKSwgY291bnRzLCByYXRlcyk6CiAgICAgICAgICAgIGF4LnRleHQocmF0ZSArIDEuMCwgeSwgZiJ7Y291bnQ6LH0gKHtyYXRlOi4xZn0lKSIsIHZhPSJjZW50ZXIiLCBmb250c2l6ZT0xMCwgY29sb3I9IiMyNjM1NDIiKQogICAgZm9vdGVyKGZpZykKICAgIGV4cG9ydChmaWcsIG91dHB1dCwgInNjYWlfaG91cjE2X2Rpc3RyaWJ1dGlvbnMiKQoKICAgIGZpZywgYXhlcyA9IHBsdC5zdWJwbG90cygxLCAzLCBmaWdzaXplPSgxNywgOS41KSkKICAgIGZpZy5zdWJwbG90c19hZGp1c3QobGVmdD0wLjE0NSwgcmlnaHQ9MC45OSwgdG9wPTAuNzQsIGJvdHRvbT0wLjIwLCB3c3BhY2U9MC4xMSkKICAgIGZpZy50ZXh0KDAuMDM1LCAwLjk1LCAiSG91ci0xNiBTQ0FJIHByb3h5IGJ5IGVhcmx5IFNwT+KCgiBpbnN0YWJpbGl0eSIsIGZvbnRzaXplPTIyLCB3ZWlnaHQ9ImJvbGQiLCBjb2xvcj0iIzI2MzU0MiIpCiAgICBmaWcudGV4dCgwLjAzNSwgMC45MDUsICJEZXNjcmlwdGl2ZSwgdW5hZGp1c3RlZCBjb21wYXJpc29uIHdpdGhpbiBlYWNoIHNhbXBsZTsgbm8gaW5mZXJlbmNlIG9mIGNhdXNhdGlvbiBvciBwcmVkaWN0aXZlIHBlcmZvcm1hbmNlLiIsIGZvbnRzaXplPTEyLCBjb2xvcj0iIzUyNjA2RCIpCiAgICBoYW5kbGVzID0gW3BsdC5SZWN0YW5nbGUoKDAsIDApLCAxLCAxLCBjb2xvcj0iIzI2NzZBRCIpLCBwbHQuUmVjdGFuZ2xlKCgwLCAwKSwgMSwgMSwgY29sb3I9IiM4ODk3QTQiKV0KICAgIGZpZy5sZWdlbmQoaGFuZGxlcywgWyJFYXJseSBpbnN0YWJpbGl0eSIsICJObyBlYXJseSBpbnN0YWJpbGl0eSJdLCBsb2M9InVwcGVyIGxlZnQiLCBiYm94X3RvX2FuY2hvcj0oMC4wMywgMC44OCksIGZyYW1lb249RmFsc2UsIG5jb2w9MiwgZm9udHNpemU9MTEpCiAgICBmb3IgaSwgKHBvcHVsYXRpb24sIHRpdGxlKSBpbiBlbnVtZXJhdGUoUE9QVUxBVElPTlMuaXRlbXMoKSk6CiAgICAgICAgYXggPSBheGVzW2ldCiAgICAgICAgc3R5bGVfYXhpcyhheCwgaSkKICAgICAgICB0b3RhbF9uLCB0b3RhbF9jb3VudHMgPSBjb3VudHNfZm9yKGZyYW1lLCBwb3B1bGF0aW9uLCAiYWxsIikKICAgICAgICBuMSwgYzEgPSBjb3VudHNfZm9yKGZyYW1lLCBwb3B1bGF0aW9uLCAiaW5zdGFiaWxpdHkiKQogICAgICAgIG4wLCBjMCA9IGNvdW50c19mb3IoZnJhbWUsIHBvcHVsYXRpb24sICJub19pbnN0YWJpbGl0eSIpCiAgICAgICAgaWYgbjEgKyBuMCAhPSB0b3RhbF9uIG9yIG5vdCBucC5hcnJheV9lcXVhbChjMSArIGMwLCB0b3RhbF9jb3VudHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRXhwb3N1cmUgZ3JvdXBzIGRvIG5vdCByZWNvbmNpbGU6IHtwb3B1bGF0aW9ufSIpCiAgICAgICAgYXguc2V0X3RpdGxlKGYie3RpdGxlfVxuSW5zdGFiaWxpdHkgbiA9IHtuMTosfTsgbm8gaW5zdGFiaWxpdHkgbiA9IHtuMDosfSIsIGxvYz0ibGVmdCIsIGZvbnRzaXplPTExLCB3ZWlnaHQ9ImJvbGQiLCBwYWQ9MjApCiAgICAgICAgZm9yIG4sIGNvdW50cywgb2Zmc2V0LCBjb2xvciBpbiBbKG4xLCBjMSwgLTAuMTgsICIjMjY3NkFEIiksIChuMCwgYzAsIDAuMTgsICIjODg5N0E0IildOgogICAgICAgICAgICByYXRlcyA9IGNvdW50cyAvIG4gKiAxMDAKICAgICAgICAgICAgeXMgPSBucC5hcmFuZ2UobGVuKGNvdW50cykpICsgb2Zmc2V0CiAgICAgICAgICAgIGF4LmJhcmgoeXMsIHJhdGVzLCBoZWlnaHQ9MC4zMSwgY29sb3I9Y29sb3IpCiAgICAgICAgICAgIGZvciB5LCBjb3VudCwgcmF0ZSBpbiB6aXAoeXMsIGNvdW50cywgcmF0ZXMpOgogICAgICAgICAgICAgICAgYXgudGV4dChyYXRlICsgMS4wLCB5LCBmIntjb3VudDosfSAoe3JhdGU6LjFmfSUpIiwgdmE9ImNlbnRlciIsIGZvbnRzaXplPTksIGNvbG9yPSIjMjYzNTQyIikKICAgIGZvb3RlcihmaWcsIHN0cmF0aWZpZWQ9VHJ1ZSkKICAgIGV4cG9ydChmaWcsIG91dHB1dCwgInNjYWlfaG91cjE2X2J5X2luc3RhYmlsaXR5IikKICAgIGF1ZGl0ID0geyJzb3VyY2UiOiBzb3VyY2UubmFtZSwgInNvdXJjZV9zaGEyNTYiOiBoYXNobGliLnNoYTI1Nihzb3VyY2UucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwgIm1lYXN1cmVtZW50X2xvb2tiYWNrX21pbnV0ZXMiOiAyNDAsICJwZXJjZW50X2Rlbm9taW5hdG9yIjogImVudGlyZSBwb3B1bGF0aW9uL2V4cG9zdXJlIGdyb3VwLCBpbmNsdWRpbmcgdW5jbGFzc2lmaWVkIGFuZCBub3QtaW4tSUNVIHN0YXlzIiwgInN0YWdlX0EiOiAidW5hdmFpbGFibGU7IG5vdCBwbG90dGVkIGFzIHplcm8iLCAiZGVwYXJ0dXJlX2Rpc3BsYXkiOiAic3VtIG9mIG11dHVhbGx5IGV4Y2x1c2l2ZSBkaXNjaGFyZ2VkX2J5X2hvdXIxNiBhbmQgbGVmdF9pY3VfYnlfaG91cjE2IiwgImV4ZWN1dGlvbiI6ICJsb2NhbCBhZ2dyZWdhdGUgcGxvdHRpbmc7IG5vIGNsaW5pY2FsIGRhdGEgcHJvY2Vzc2luZyBvciBtb2RlbCByZXJ1biIsICJmaWd1cmVzIjogWyJzY2FpX2hvdXIxNl9kaXN0cmlidXRpb25zIiwgInNjYWlfaG91cjE2X2J5X2luc3RhYmlsaXR5Il0sICJmb3JtYXRzIjogWyJwbmciLCAicGRmIiwgInN2ZyJdfQogICAgKG91dHB1dCAvICJmaWd1cmVfYXVkaXQuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhhdWRpdCwgaW5kZW50PTIpICsgIlxuIikKICAgIHByaW50KGYiU2F2ZWQgdHdvIGZpZ3VyZXMgaW4gUE5HLCBQREYsIGFuZCBTVkc6IHtvdXRwdXR9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="))
subprocess.check_call([sys.executable, str(SCAI_PLOT_SCRIPT), "--results-dir", str(SCAI_RESULTS)])
for stem in ("scai_hour16_distributions", "scai_hour16_by_instability"):
    display(Image(filename=str(SCAI_RESULTS / "figures" / f"{stem}.png"), width=1200))
